In [1]:
import os, sys
os.chdir("/workspace/finetune/dwarf")
sys.path.insert(0, os.getcwd())

In [2]:
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase
from langchain_community.chat_models import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import ChatHuggingFace

from scripts.schema_converter import convert_schema_text

/root/miniconda3/envs/py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
engine = create_engine("sqlite:///database/__movie_information_and_analysis__.sqlite")

In [4]:
database = SQLDatabase(
    engine, # connection to the database
    sample_rows_in_table_info=3 # number of rows to show in table info
)

In [5]:
prompt_text_to_sql = """
You are an expert SQL assistant. Use ONLY the provided table schema to write a correct SQL query. Return valid SQL without explanations.
Table schema:
{table_info}
Question: {input}
"""

prompt_template_text_to_sql = PromptTemplate(
    template=prompt_text_to_sql,
    input_variables=["table_info", "input"],
)

In [6]:
def get_schema(_):
    table_info_original = database.get_table_info()
    table_info = convert_schema_text(table_info_original)
    return table_info

print("Schema:", get_schema({}))

Schema: CREATE TABLE casts (
	cast_id INTEGER, 
	movie_id INTEGER, 
	actor_name TEXT, 
	character_name TEXT, 
	character_gender TEXT, 
	character_age INTEGER, 
	actor_birthday TEXT, 
	actor_bio TEXT, 
	actor_image_url TEXT, 
	actor_imdb_id TEXT, 
	character_rank INTEGER, 
	PRIMARY KEY (cast_id), 
	CONSTRAINT fk_casts_movie_id FOREIGN KEY(movie_id) REFERENCES movies (movie_id)
);
INSERT INTO casts (cast_id, movie_id, actor_name, character_name, character_gender, character_age, actor_birthday, actor_bio, actor_image_url, actor_imdb_id, character_rank) VALUES (0, 0, 'Chris Pratt', 'Owen Grady', 'Male', 32, '1979-06-21', 'Chris Pratt is an American actor known for his roles in Guardians of the Galaxy and Jurassic World.', 'http://example.com/actor1.jpg', 'nm0000000', 1), (1, 1, 'Bryce Dallas Howard', 'Claire Dearing', 'Female', 34, '1981-03-02', 'Bryce Dallas Howard is an American actress known for her roles in Jurassic World and The Help.', 'http://example.com/actor2.jpg', 'nm0000001', 2)

In [7]:
def get_response(query):
    response = database.run(query)
    return response

print("Response:", get_response("SELECT * FROM movies LIMIT 5"))

Response: [(0, 'tt0369610', 150000000, 1513528810, 'Jurassic World', 'The park is open.', 'Twenty-two years after the events of Jurassic Park...', 124, '2015-06-09', 2015, 'http://example.com/poster1.jpg', 'http://example.com/trailer1.mp4', 'USA', 'English', 'PG-13', '150000000 (Production), 50000000 (Marketing)', 500000000, 1013528810, 1513528810, 'USD', 'Released', 124), (1, 'tt1392190', 150000000, 378436354, 'Mad Max: Fury Road', 'What a Lovely Day.', 'An apocalyptic story set in the furthest reaches...', 120, '2015-05-13', 2015, 'http://example.com/poster2.jpg', 'http://example.com/trailer2.mp4', 'Australia', 'English', 'R', '150000000 (Production), 50000000 (Marketing)', 100000000, 278436354, 378436354, 'USD', 'Released', 120)]


In [41]:
import re
def post_process_sql(sql):
    # get text after "<start_of_turn>model"
    pattern = r"<start_of_turn>model(.*)"
    match = re.search(pattern, sql, re.DOTALL)
    if match:
        sql = match.group(1).strip()
        # ambil text start select sampai sebelum ;
        pattern = r"(SELECT.*?);"
        match = re.search(pattern, sql, re.DOTALL | re.IGNORECASE)
        if match:
            sql = match.group(1).strip()
            return sql
    return None

In [9]:
model_path = "/workspace/finetune/dwarf/outputs/gemma-3-1b-sql-qlora/merged"

In [10]:
from langchain_huggingface.llms import HuggingFacePipeline

In [ ]:
model = HuggingFacePipeline.from_model_id(
    model_id = model_path,
    task="text-generation",
    model_kwargs={"trust_remote_code": True, "device_map":"auto"},
)

Device set to use cuda:0


In [35]:
chat = ChatHuggingFace(
    llm=model,
    temperature=0,
)

In [36]:
text = """You are an expert SQL assistant. Use ONLY the provided table schema to write a correct SQL query. Return valid SQL without explanations.
Table schema:
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
Question: What is the total volume of timber sold by each salesperson, sorted by salesperson?"""
res = chat.invoke(text)

In [37]:
print(res.content)

<bos><start_of_turn>user
You are an expert SQL assistant. Use ONLY the provided table schema to write a correct SQL query. Return valid SQL without explanations.
Table schema:
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
Question: What is the total volume of timber sold by each salesperson, sorted by salesperson?<end_of_turn>
<start_of_turn>model
SELECT s.name, SUM(ts.volume) as total_volume FROM salesperson s JOIN timber_sales ts ON s.salesperson_id = ts.salesperson_id GROUP BY s.name ORDER BY total_volume DESC;


In [42]:
sql_chain_response = (
    RunnablePassthrough.assign(
        table_info=get_schema,
    )
    | prompt_template_text_to_sql
    | chat
    | StrOutputParser()
)

def sql_generator(query: str) -> str:
    sql = sql_chain_response.invoke({"input": query})
    sql = post_process_sql(sql)
    return sql

In [46]:
sql_response = sql_generator("How many casting person in database?")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [47]:
print(sql_response)

SELECT COUNT(*) FROM casts


In [49]:
database.run(sql_response)

'[(2,)]'